# Preprocessing, Train/Test Split & Feature Engineering

This notebook takes the analysis-ready NHANES dataset produced by `02_data_preparation.ipynb` and prepares it for model training. It is the direct input pipeline for Logistic Regression, Decision Tree, Random Forest, and XGBoost.

The pipeline applies the following steps in order:

1. **Load** the analysis-ready parquet file.
2. **Feature audit** — report missingness, dtypes, and class balance before any transformation.
3. **Feature typing** — explicitly separate continuous, ordinal, and nominal (binary/multi-category) features.
4. **Train/test split** — stratified 80/20 split; `CYCLE` is used for a secondary temporal leakage check.
5. **Imputation** — median for continuous and ordinal features; most-frequent for nominal features. Fitted on train only, applied to both.
6. **Encoding** — ordinal features label-encoded in place; nominal features one-hot encoded.
7. **Scaling** — StandardScaler fitted on train only. Applied to continuous + ordinal columns for models that need it (LR); tree-based models receive the unscaled version.
8. **Save** four split files and two full preprocessed files (scaled / unscaled).

**Output files** (written to `data/splits/`):

| File | Contents |
|---|---|
| `X_train_scaled.parquet` | Training features — scaled (for Logistic Regression) |
| `X_test_scaled.parquet` | Test features — scaled (for Logistic Regression) |
| `X_train_unscaled.parquet` | Training features — unscaled (for tree-based models) |
| `X_test_unscaled.parquet` | Test features — unscaled (for tree-based models) |
| `y_train.parquet` | Training labels |
| `y_test.parquet` | Test labels |
| `feature_names.json` | Ordered list of feature columns after encoding |
| `preprocessing_report.txt` | Audit log of all steps |

**Note:** This notebook reads `data/processed/nhanes_analysis_ready.parquet` produced by `02_data_preparation.ipynb`. Run that notebook first if the file does not exist.

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
TEST_SIZE    = 0.20

## Feature Type Definitions

Each feature is assigned to exactly one of three groups:

| Group | Treatment | Examples |
|---|---|---|
| `CONTINUOUS` | Median imputation → StandardScaler | age, BMI, cholesterol, lab values |
| `ORDINAL` | Median imputation → label-encoded → StandardScaler | education level, self-rated health, healthcare visits |
| `NOMINAL_BINARY` | Mode imputation → passed through as-is (already 1/2 coded) | sex, ever smoked, insurance |
| `NOMINAL_MULTI` | Mode imputation → one-hot encoded | race/ethnicity, food security category |

Binary NHANES survey variables are coded 1/2 (Yes/No). These are treated as nominal binary and are **not** scaled, keeping their interpretability for feature importance analysis.

In [2]:
# ── Continuous features (median impute → scale) ───────────────────────────────
CONTINUOUS_FEATURES = [
    "age",
    "income_poverty_ratio",
    "bmi",
    "waist_circumference",
    "mean_systolic_bp",
    "mean_diastolic_bp",
    "sedentary_minutes_per_day",
    "physical_activity_moderate_equivalent_min_week",
    "sleep_hours",
    "phq9_score",
    "total_cholesterol_mg_dl",
    "hdl_cholesterol_mg_dl",
    "energy_kcal",
    "protein_g",
    "carbohydrate_g",
    "total_sugar_g",
    "fiber_g",
    "total_fat_g",
    "cholesterol_mg",
]

# ── Ordinal features (median impute → integer label → scale) ─────────────────
# Encoded in natural order (low → high). Order lists define the encoding.
ORDINAL_FEATURES = [
    "education_level",           # 1=<9th grade … 5=college grad
    "self_rated_health",         # 1=Excellent … 5=Poor
    "healthcare_visits_past_12_months",  # 0=None, 1=1, 2=2–3, 3=4–5, 4=6–7, 5=8–9, 6=10+
    "adult_food_security_category",     # 1=Full … 4=Very low
    "household_food_security_category", # 1=Full … 4=Very low
]

# ── Binary nominal features (mode impute, keep as-is) ────────────────────────
# All coded 1=Yes / 2=No in NHANES. Left as numeric; no OHE needed.
NOMINAL_BINARY_FEATURES = [
    "sex",                                      # 1=Male, 2=Female
    "family_history_diabetes",                  # 1=Yes, 2=No
    "ever_smoked_100_cigarettes",               # 1=Yes, 2=No
    "ever_almost_daily_heavy_drinking",         # 1=Yes, 2=No
    "vigorous_work_activity",                   # 1=Yes, 2=No
    "moderate_work_activity",                   # 1=Yes, 2=No
    "walk_or_bicycle_transport",                # 1=Yes, 2=No
    "vigorous_recreational_activity",           # 1=Yes, 2=No
    "moderate_recreational_activity",           # 1=Yes, 2=No
    "trouble_sleeping_reported_to_doctor",      # 1=Yes, 2=No
    "ever_told_hypertension",                   # 1=Yes, 2=No
    "ever_told_high_cholesterol",               # 1=Yes, 2=No
    "has_health_insurance",                     # 1=Yes, 2=No
    "insurance_gap_past_12_months",             # 1=Yes, 2=No
    "usual_place_for_healthcare",               # 1=Yes, 2=No (has a usual place)
]

# ── Multi-category nominal features (mode impute → one-hot encode) ────────────
NOMINAL_MULTI_FEATURES = [
    "race_ethnicity",  # 1=Mexican American, 2=Other Hispanic, 3=Non-Hispanic White,
                       # 4=Non-Hispanic Black, 6=Non-Hispanic Asian, 7=Other/Multiracial
]

# All features in a single ordered list (used for consistency checks)
ALL_FEATURES = (
    CONTINUOUS_FEATURES
    + ORDINAL_FEATURES
    + NOMINAL_BINARY_FEATURES
    + NOMINAL_MULTI_FEATURES
)

## 1. Load the Analysis-Ready Dataset

In [3]:
input_path = Path("../data/processed/nhanes_analysis_ready.parquet")
df = pd.read_parquet(input_path)

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print()
print("Cycle distribution:")
print(df["cycle"].value_counts().sort_index())

Loaded: 26,000 rows x 43 columns

Cycle distribution:
cycle
2011-2012          5502
2013-2014          5704
2015-2016          5649
2017-March 2020    9145
Name: count, dtype: int64


## 2. Feature Audit (Pre-Processing)

Inspect missingness, data types, and class balance before any transformations are applied. This forms the baseline for the preprocessing report.

In [4]:
# ── Class balance ─────────────────────────────────────────────────────────────
print("Outcome (diabetes) distribution:")
print(df["diabetes"].value_counts(dropna=False))
print()
print(df["diabetes"].value_counts(normalize=True, dropna=True).round(3))

Outcome (diabetes) distribution:
diabetes
0    21091
1     4909
Name: count, dtype: Int64

diabetes
0    0.811
1    0.189
Name: proportion, dtype: Float64


In [5]:
# ── Missingness audit ─────────────────────────────────────────────────────────
missing_audit = (
    df[ALL_FEATURES]
    .isnull()
    .sum()
    .rename("n_missing")
    .to_frame()
)
missing_audit["pct_missing"] = (missing_audit["n_missing"] / len(df) * 100).round(1)
missing_audit = missing_audit[missing_audit["n_missing"] > 0].sort_values("pct_missing", ascending=False)

print(f"{len(missing_audit)} of {len(ALL_FEATURES)} features have missing values:\n")
print(missing_audit.to_string())

38 of 40 features have missing values:

                                                n_missing  pct_missing
physical_activity_moderate_equivalent_min_week       8555         32.9
ever_almost_daily_heavy_drinking                     6579         25.3
healthcare_visits_past_12_months                     5905         22.7
insurance_gap_past_12_months                         4944         19.0
protein_g                                            3679         14.2
cholesterol_mg                                       3679         14.2
total_fat_g                                          3679         14.2
fiber_g                                              3679         14.2
total_sugar_g                                        3679         14.2
carbohydrate_g                                       3679         14.2
energy_kcal                                          3679         14.2
phq9_score                                           3662         14.1
income_poverty_ratio                 

In [6]:
# ── Check for any features defined above that are absent from the dataset ─────
missing_cols = [c for c in ALL_FEATURES if c not in df.columns]
if missing_cols:
    print(f"WARNING: {len(missing_cols)} defined features are absent from the dataset:")
    for c in missing_cols:
        print(f"  - {c}")
else:
    print("All defined features are present in the dataset. ✓")

All defined features are present in the dataset. ✓


## 3. Prepare Feature Matrix and Target Vector

In [7]:
# Only keep features that are actually present
available_features = [c for c in ALL_FEATURES if c in df.columns]

X = df[available_features].copy()
y = df["diabetes"].astype(int).copy()

# Update group lists to only include available columns
continuous_cols      = [c for c in CONTINUOUS_FEATURES      if c in available_features]
ordinal_cols         = [c for c in ORDINAL_FEATURES         if c in available_features]
nominal_binary_cols  = [c for c in NOMINAL_BINARY_FEATURES  if c in available_features]
nominal_multi_cols   = [c for c in NOMINAL_MULTI_FEATURES   if c in available_features]

print(f"Feature matrix:  {X.shape[0]:,} rows x {X.shape[1]} columns")
print(f"Target vector:   {y.shape[0]:,} rows")
print()
print(f"  Continuous features:      {len(continuous_cols)}")
print(f"  Ordinal features:         {len(ordinal_cols)}")
print(f"  Nominal binary features:  {len(nominal_binary_cols)}")
print(f"  Nominal multi features:   {len(nominal_multi_cols)}")

Feature matrix:  26,000 rows x 40 columns
Target vector:   26,000 rows

  Continuous features:      19
  Ordinal features:         5
  Nominal binary features:  15
  Nominal multi features:   1


## 4. Stratified Train / Test Split

An **80 / 20 stratified split** is applied, stratifying on the `diabetes` label to preserve class proportions in both partitions.

A temporal leakage check is then printed: because the NHANES cycles span 2011–2020, it is important to verify that no single cycle is entirely absent from the training set or entirely isolated in the test set.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Train set: {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:  {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.1f}%)")
print()
print("Class balance — train:")
print(y_train.value_counts(normalize=True).round(3))
print()
print("Class balance — test:")
print(y_test.value_counts(normalize=True).round(3))

Train set: 20,800 rows (80.0%)
Test set:  5,200 rows (20.0%)

Class balance — train:
diabetes
0    0.811
1    0.189
Name: proportion, dtype: float64

Class balance — test:
diabetes
0    0.811
1    0.189
Name: proportion, dtype: float64


In [9]:
# ── Temporal leakage check ────────────────────────────────────────────────────
cycles = df.loc[X_train.index, "cycle"]
cycles_test = df.loc[X_test.index, "cycle"]

cycle_summary = pd.DataFrame({
    "train_n":   cycles.value_counts().sort_index(),
    "test_n":    cycles_test.value_counts().sort_index(),
}).fillna(0).astype(int)

cycle_summary["train_pct"] = (cycle_summary["train_n"] / cycle_summary["train_n"].sum() * 100).round(1)
cycle_summary["test_pct"]  = (cycle_summary["test_n"]  / cycle_summary["test_n"].sum()  * 100).round(1)

print("Cycle distribution across train / test splits:")
print(cycle_summary.to_string())
print()
print("Note: Stratified split is by outcome, not by cycle. All cycles should")
print("appear in both partitions, confirming no temporal leakage.")

Cycle distribution across train / test splits:
                 train_n  test_n  train_pct  test_pct
cycle                                                
2011-2012           4403    1099       21.2      21.1
2013-2014           4591    1113       22.1      21.4
2015-2016           4515    1134       21.7      21.8
2017-March 2020     7291    1854       35.1      35.7

Note: Stratified split is by outcome, not by cycle. All cycles should
appear in both partitions, confirming no temporal leakage.


## 5. Imputation

Imputers are **fitted on the training set only** and then applied to both train and test sets, preventing any information leakage from the test set.

| Feature group | Strategy | Rationale |
|---|---|---|
| Continuous | Median | Robust to skewed distributions (BMI, lab values) |
| Ordinal | Median (rounded to nearest integer) | Preserves ordered category structure |
| Nominal (binary & multi) | Most frequent | Appropriate for categorical variables |

In [10]:
# ── Continuous imputation ─────────────────────────────────────────────────────
if continuous_cols:
    imputer_continuous = SimpleImputer(strategy="median")
    X_train[continuous_cols] = imputer_continuous.fit_transform(X_train[continuous_cols])
    X_test[continuous_cols]  = imputer_continuous.transform(X_test[continuous_cols])
    print(f"Continuous imputation (median) applied to {len(continuous_cols)} columns.")

# ── Ordinal imputation ────────────────────────────────────────────────────────
if ordinal_cols:
    imputer_ordinal = SimpleImputer(strategy="median")
    X_train[ordinal_cols] = imputer_ordinal.fit_transform(X_train[ordinal_cols])
    X_test[ordinal_cols]  = imputer_ordinal.transform(X_test[ordinal_cols])
    # Round to nearest integer to keep valid ordinal category codes
    X_train[ordinal_cols] = X_train[ordinal_cols].round().astype(int)
    X_test[ordinal_cols]  = X_test[ordinal_cols].round().astype(int)
    print(f"Ordinal imputation (median, rounded) applied to {len(ordinal_cols)} columns.")

# ── Nominal binary imputation ─────────────────────────────────────────────────
if nominal_binary_cols:
    imputer_nominal_bin = SimpleImputer(strategy="most_frequent")
    X_train[nominal_binary_cols] = imputer_nominal_bin.fit_transform(X_train[nominal_binary_cols])
    X_test[nominal_binary_cols]  = imputer_nominal_bin.transform(X_test[nominal_binary_cols])
    print(f"Nominal binary imputation (mode) applied to {len(nominal_binary_cols)} columns.")

# ── Nominal multi-category imputation ─────────────────────────────────────────
if nominal_multi_cols:
    imputer_nominal_multi = SimpleImputer(strategy="most_frequent")
    X_train[nominal_multi_cols] = imputer_nominal_multi.fit_transform(X_train[nominal_multi_cols])
    X_test[nominal_multi_cols]  = imputer_nominal_multi.transform(X_test[nominal_multi_cols])
    print(f"Nominal multi imputation (mode) applied to {len(nominal_multi_cols)} columns.")

# Verify no missing values remain
remaining_missing_train = X_train.isnull().sum().sum()
remaining_missing_test  = X_test.isnull().sum().sum()
print()
print(f"Remaining missing values — train: {remaining_missing_train}  |  test: {remaining_missing_test}")

Continuous imputation (median) applied to 19 columns.
Ordinal imputation (median, rounded) applied to 5 columns.
Nominal binary imputation (mode) applied to 15 columns.
Nominal multi imputation (mode) applied to 1 columns.

Remaining missing values — train: 0  |  test: 0


## 6. Encoding

**Ordinal features** are already integer-coded after imputation (NHANES codes align with their natural order), so no further encoding is needed beyond confirming correct dtype.

**Multi-category nominal features** are one-hot encoded. The first category is dropped (`drop='first'`) to avoid the dummy variable trap for linear models. Tree-based models are unaffected by this choice. The encoder is fitted on train only.

In [11]:
ohe_feature_names = []

if nominal_multi_cols:
    # One-hot encode multi-category nominal features
    X_train_ohe = pd.get_dummies(
        X_train[nominal_multi_cols].astype(str),
        drop_first=True,
        prefix={c: c for c in nominal_multi_cols},
    )
    # Apply the same columns to the test set (align ensures no unseen categories cause issues)
    X_test_ohe = pd.get_dummies(
        X_test[nominal_multi_cols].astype(str),
        drop_first=True,
        prefix={c: c for c in nominal_multi_cols},
    ).reindex(columns=X_train_ohe.columns, fill_value=0)

    ohe_feature_names = list(X_train_ohe.columns)

    # Drop original nominal multi columns and append encoded columns
    X_train = X_train.drop(columns=nominal_multi_cols)
    X_test  = X_test.drop(columns=nominal_multi_cols)

    X_train = pd.concat([X_train.reset_index(drop=True), X_train_ohe.reset_index(drop=True)], axis=1)
    X_test  = pd.concat([X_test.reset_index(drop=True),  X_test_ohe.reset_index(drop=True)],  axis=1)

    print(f"One-hot encoded {len(nominal_multi_cols)} column(s) → {len(ohe_feature_names)} dummy column(s):")
    for col in ohe_feature_names:
        print(f"  {col}")
else:
    print("No multi-category nominal features to encode.")

print()
print(f"Feature matrix shape after encoding — train: {X_train.shape}  |  test: {X_test.shape}")

One-hot encoded 1 column(s) → 4 dummy column(s):
  race_ethnicity_2.0
  race_ethnicity_3.0
  race_ethnicity_4.0
  race_ethnicity_6.0

Feature matrix shape after encoding — train: (20800, 43)  |  test: (5200, 43)


## 7. Scaling

A **StandardScaler** (zero mean, unit variance) is fitted on the training set only and applied to continuous and ordinal columns for the **scaled** version of the dataset used by Logistic Regression.

Tree-based models (Random Forest, Decision Tree, XGBoost) do not require scaling. An **unscaled** version of the feature matrix is saved separately, keeping identical imputation and encoding but omitting the scaling step. This ensures fair comparison of feature importance magnitudes across model types.

In [12]:
# Columns to scale = continuous + ordinal (both present and remaining after encoding)
cols_to_scale = [c for c in continuous_cols + ordinal_cols if c in X_train.columns]

# ── Unscaled copies (for tree-based models) ───────────────────────────────────
X_train_unscaled = X_train.copy()
X_test_unscaled  = X_test.copy()

# ── Scaled copies (for Logistic Regression) ───────────────────────────────────
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

scaler = StandardScaler()
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train_scaled[cols_to_scale])
X_test_scaled[cols_to_scale]  = scaler.transform(X_test_scaled[cols_to_scale])

print(f"StandardScaler applied to {len(cols_to_scale)} columns:")
for c in cols_to_scale:
    print(f"  {c}")

StandardScaler applied to 24 columns:
  age
  income_poverty_ratio
  bmi
  waist_circumference
  mean_systolic_bp
  mean_diastolic_bp
  sedentary_minutes_per_day
  physical_activity_moderate_equivalent_min_week
  sleep_hours
  phq9_score
  total_cholesterol_mg_dl
  hdl_cholesterol_mg_dl
  energy_kcal
  protein_g
  carbohydrate_g
  total_sugar_g
  fiber_g
  total_fat_g
  cholesterol_mg
  education_level
  self_rated_health
  healthcare_visits_past_12_months
  adult_food_security_category
  household_food_security_category


## 8. Post-Processing Validation

Sanity checks before writing output files.

In [13]:
print("=" * 60)
print("POST-PROCESSING VALIDATION")
print("=" * 60)

# Shape checks
print(f"\nX_train_scaled:   {X_train_scaled.shape}")
print(f"X_test_scaled:    {X_test_scaled.shape}")
print(f"X_train_unscaled: {X_train_unscaled.shape}")
print(f"X_test_unscaled:  {X_test_unscaled.shape}")
print(f"y_train:          {y_train.shape}")
print(f"y_test:           {y_test.shape}")

# Missing value check
assert X_train_scaled.isnull().sum().sum()   == 0, "Missing values in X_train_scaled!"
assert X_test_scaled.isnull().sum().sum()    == 0, "Missing values in X_test_scaled!"
assert X_train_unscaled.isnull().sum().sum() == 0, "Missing values in X_train_unscaled!"
assert X_test_unscaled.isnull().sum().sum()  == 0, "Missing values in X_test_unscaled!"
print("\n✓ No missing values in any feature matrix.")

# Column consistency
assert list(X_train_scaled.columns) == list(X_test_scaled.columns),     "Column mismatch: scaled"
assert list(X_train_unscaled.columns) == list(X_test_unscaled.columns), "Column mismatch: unscaled"
assert list(X_train_scaled.columns) == list(X_train_unscaled.columns),  "Column mismatch: scaled vs unscaled"
print("✓ Column order consistent across all splits.")

# Class balance
print("\nClass balance:")
print(f"  y_train:  {dict(y_train.value_counts(normalize=True).round(3))}")
print(f"  y_test:   {dict(y_test.value_counts(normalize=True).round(3))}")

# Scaled vs unscaled: same columns, different values for numeric cols
print("\nSample: mean of BMI column (continuous) to confirm scaling:")
if "bmi" in X_train_scaled.columns:
    print(f"  Scaled   — train mean: {X_train_scaled['bmi'].mean():.4f}  (should be ~0)")
    print(f"  Unscaled — train mean: {X_train_unscaled['bmi'].mean():.2f}")

print("\n✓ All validation checks passed.")

POST-PROCESSING VALIDATION

X_train_scaled:   (20800, 43)
X_test_scaled:    (5200, 43)
X_train_unscaled: (20800, 43)
X_test_unscaled:  (5200, 43)
y_train:          (20800,)
y_test:           (5200,)

✓ No missing values in any feature matrix.
✓ Column order consistent across all splits.

Class balance:
  y_train:  {0: np.float64(0.811), 1: np.float64(0.189)}
  y_test:   {0: np.float64(0.811), 1: np.float64(0.189)}

Sample: mean of BMI column (continuous) to confirm scaling:
  Scaled   — train mean: -0.0000  (should be ~0)
  Unscaled — train mean: 29.35

✓ All validation checks passed.


## 9. Save Outputs

In [14]:
output_dir = Path("../data/splits")
output_dir.mkdir(parents=True, exist_ok=True)

# ── Feature matrices ──────────────────────────────────────────────────────────
X_train_scaled.to_parquet(output_dir / "X_train_scaled.parquet",   index=False)
X_test_scaled.to_parquet(output_dir  / "X_test_scaled.parquet",    index=False)
X_train_unscaled.to_parquet(output_dir / "X_train_unscaled.parquet", index=False)
X_test_unscaled.to_parquet(output_dir  / "X_test_unscaled.parquet",  index=False)

# ── Target vectors ────────────────────────────────────────────────────────────
y_train.reset_index(drop=True).to_frame(name="diabetes").to_parquet(output_dir / "y_train.parquet", index=False)
y_test.reset_index(drop=True).to_frame(name="diabetes").to_parquet(output_dir / "y_test.parquet",   index=False)

# ── Feature name manifest ─────────────────────────────────────────────────────
feature_names = list(X_train_scaled.columns)
with open(output_dir / "feature_names.json", "w") as f:
    json.dump(feature_names, f, indent=2)

print(f"Saved to: {output_dir.resolve()}")
print()
for fname in sorted(output_dir.iterdir()):
    size_kb = fname.stat().st_size / 1024
    print(f"  {fname.name:<45} {size_kb:>7.1f} KB")

Saved to: /Users/kaylandegroot/Desktop/G2_Feature_Importance_DT2_Prediction/data/splits

  X_test_scaled.parquet                           353.5 KB
  X_test_unscaled.parquet                         276.0 KB
  X_train_scaled.parquet                         1125.7 KB
  X_train_unscaled.parquet                        920.3 KB
  feature_names.json                                1.1 KB
  y_test.parquet                                    2.0 KB
  y_train.parquet                                   4.2 KB


In [15]:
# ── Preprocessing report ──────────────────────────────────────────────────────
report_lines = [
    "NHANES Diabetes Prediction — Preprocessing Report",
    "=" * 55,
    "",
    f"Input file:           {input_path}",
    f"Total samples:        {len(df):,}",
    f"Train samples:        {len(X_train_scaled):,}",
    f"Test samples:         {len(X_test_scaled):,}",
    f"Test size:            {TEST_SIZE*100:.0f}%",
    f"Random state:         {RANDOM_STATE}",
    f"Stratified split:     Yes (by diabetes label)",
    "",
    "Feature counts after encoding:",
    f"  Total features:     {len(feature_names)}",
    f"  Continuous:         {len(continuous_cols)}",
    f"  Ordinal:            {len(ordinal_cols)}",
    f"  Nominal binary:     {len(nominal_binary_cols)}",
    f"  OHE dummies:        {len(ohe_feature_names)}",
    "",
    "Imputation:",
    f"  Continuous:         Median",
    f"  Ordinal:            Median (rounded to int)",
    f"  Nominal:            Most frequent",
    "",
    "Scaling:",
    f"  Method:             StandardScaler (mean=0, std=1)",
    f"  Columns scaled:     {len(cols_to_scale)} (continuous + ordinal)",
    f"  Fit on:             Training set only",
    "",
    "Output files:",
    "  X_train_scaled.parquet   → Logistic Regression",
    "  X_test_scaled.parquet    → Logistic Regression",
    "  X_train_unscaled.parquet → Decision Tree, Random Forest, XGBoost",
    "  X_test_unscaled.parquet  → Decision Tree, Random Forest, XGBoost",
    "  y_train.parquet          → All models",
    "  y_test.parquet           → All models",
    "  feature_names.json       → Feature name manifest",
    "",
    "Class distribution (train):",
    f"  No diabetes (0): {y_train.value_counts(normalize=True)[0]:.3f}",
    f"  Diabetes (1):    {y_train.value_counts(normalize=True)[1]:.3f}",
    "",
    "Class distribution (test):",
    f"  No diabetes (0): {y_test.value_counts(normalize=True)[0]:.3f}",
    f"  Diabetes (1):    {y_test.value_counts(normalize=True)[1]:.3f}",
]

report_path = output_dir / "preprocessing_report.txt"
with open(report_path, "w") as f:
    f.write("\n".join(report_lines))

print("\n".join(report_lines))

NHANES Diabetes Prediction — Preprocessing Report

Input file:           ../data/processed/nhanes_analysis_ready.parquet
Total samples:        26,000
Train samples:        20,800
Test samples:         5,200
Test size:            20%
Random state:         42
Stratified split:     Yes (by diabetes label)

Feature counts after encoding:
  Total features:     43
  Continuous:         19
  Ordinal:            5
  Nominal binary:     15
  OHE dummies:        4

Imputation:
  Continuous:         Median
  Ordinal:            Median (rounded to int)
  Nominal:            Most frequent

Scaling:
  Method:             StandardScaler (mean=0, std=1)
  Columns scaled:     24 (continuous + ordinal)
  Fit on:             Training set only

Output files:
  X_train_scaled.parquet   → Logistic Regression
  X_test_scaled.parquet    → Logistic Regression
  X_train_unscaled.parquet → Decision Tree, Random Forest, XGBoost
  X_test_unscaled.parquet  → Decision Tree, Random Forest, XGBoost
  y_train.parquet  

## Summary

The preprocessing pipeline is complete. The following files are ready for model training:

| File | Use for |
|---|---|
| `X_train_scaled.parquet` + `y_train.parquet` | **Logistic Regression** training |
| `X_test_scaled.parquet` + `y_test.parquet` | **Logistic Regression** evaluation |
| `X_train_unscaled.parquet` + `y_train.parquet` | **Decision Tree / Random Forest / XGBoost** training |
| `X_test_unscaled.parquet` + `y_test.parquet` | **Decision Tree / Random Forest / XGBoost** evaluation |
